In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D


In [2]:
from tensorflow.keras.models import Sequential

In [3]:
import cv2
import numpy as np
import os
import imghdr

C:\Users\gurud\AppData\Local\Temp\ipykernel_32896\1771036855.py:4: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


In [4]:
data_dir="data"

In [5]:
exts=["jpg","jpeg","png"]
for image_class in os.listdir(data_dir):
    class_dir = os.path.join(data_dir, image_class)
    for image_name in os.listdir(class_dir):
        image_path = os.path.join(class_dir, image_name)
        image = cv2.imread(image_path)
        tip=imghdr.what(image_path)
        if tip not in exts:
            print("Deleted file: ",image_path)
            os.remove(image_path)

In [6]:
data=tf.keras.utils.image_dataset_from_directory(
    data_dir)

Found 752 files belonging to 6 classes.


In [7]:
model=Sequential()

In [8]:
model.add(Conv2D(16,(3,3),activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(32,(3,3),activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Flatten())
model.add(Dense(128,activation='relu'))
model.add(Dense(6,activation='softmax'))

c:\Users\gurud\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 123008)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    15,745,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,751,014 (60.09 MB)

 Trainable params: 15,751,014 (60.09 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
len(data)

24

In [11]:
train_size=130
test_size=29
val_size=20

In [12]:
data_it=data.as_numpy_iterator()
data_it.next()

(array([[[[ 58.       ,  46.       ,  46.       ],
          [ 45.       ,  33.       ,  33.       ],
          [ 65.375    ,  53.375    ,  53.375    ],
          ...,
          [  0.       ,   0.       ,   0.       ],
          [  0.       ,   0.       ,   0.       ],
          [  0.       ,   0.       ,   0.       ]],
 
         [[ 40.125    ,  28.125    ,  28.125    ],
          [ 61.453125 ,  49.453125 ,  49.453125 ],
          [ 66.84766  ,  54.847656 ,  54.847656 ],
          ...,
          [  0.       ,   0.       ,   0.       ],
          [  0.       ,   0.       ,   0.       ],
          [  0.       ,   0.       ,   0.       ]],
 
         [[ 25.       ,  14.375    ,  13.6875   ],
          [ 58.160156 ,  47.535156 ,  46.847656 ],
          [ 62.890625 ,  52.265625 ,  51.578125 ],
          ...,
          [  0.       ,   0.       ,   0.       ],
          [  0.       ,   0.       ,   0.       ],
          [  0.       ,   0.       ,   0.       ]],
 
         ...,
 
         [[ 

In [13]:
data=data.map(lambda x,y: (x/255,y))

In [14]:
train=data.take(train_size)
test=data.skip(train_size).take(test_size)
val=data.skip(train_size+test_size).take(val_size)

In [15]:
model.compile("adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [16]:
model.fit(train,epochs=5,validation_data=val)

Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 327ms/step - accuracy: 0.1862 - loss: 6.6872 
Epoch 2/5


c:\Users\gurud\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 251ms/step - accuracy: 0.2633 - loss: 1.6977
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 246ms/step - accuracy: 0.6277 - loss: 1.0793
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 245ms/step - accuracy: 0.8737 - loss: 0.5029
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 241ms/step - accuracy: 0.9707 - loss: 0.1745


In [17]:
model.save("canine_breed.h5")